# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [52]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [53]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [54]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [55]:
from openai import OpenAI
from pydantic import BaseModel
import os
import requests
from dotenv import load_dotenv

In [56]:
import os
print(os.getcwd())

c:\Users\Mariya\deploying-ai\02_activities


In [ ]:
from dotenv import load_dotenv
import os

env_path = "C:/Users/Mariya/deploying-ai/05_scr/.env"
load_dotenv(dotenv_path=env_path)


In [58]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [59]:
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [141]:
prompt = f"""
    Analyze the following document
    <document>
    {document_text}
    </document>

    Report:
    - Author
    - Title
    - Relevance: explain why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary.
   
"""


In [142]:
tone_style = "Use concise Formal Academic Writing tone"

In [143]:
system_prompt = """ 
    You are an expert AI researcher.
    
    Return only valid JSON with the following fields:

    Author
    Title
    Relevance: a statement, no longer than one paragraph.
    Summary: max 1000 tokens.
    Tone: use the following tone style: {tone_style}
    InputTokens: number of input tokens (obtain this from the response object).
    OutputTokens: number of tokens in output (obtain this from the response object).

    Do not include explanations outside JSON.
    Do not include markdown. Do not include commentary.
    """

In [144]:
response=client.responses.create(
    model="gpt-4o",
    input=prompt,
    instructions=system_prompt
)

raw_output=response.output_text
print(raw_output)

{
    "Author": "Peter F. Drucker",
    "Title": "Managing Oneself",
    "Relevance": "This article is relevant for AI professionals as it underscores the importance of self-awareness and continual self-management, crucial skills in rapidly-evolving fields like AI where adapting to new challenges and leveraging personal strengths are key to professional success.",
    "Summary": "Peter Drucker's 'Managing Oneself' delves into the imperative of personal responsibility in career management within the knowledge economy. As traditional corporate structures dissolve, individuals must assume the role of their career's CEO, identifying strengths, understanding personal values, learning styles, and optimal work environments. Drucker emphasizes feedback analysis for recognizing true strengths and addressing weaknesses, advocating for focus on enhancing innate abilities rather than fixing incompetencies. The article explores understanding personal work styles (reader vs. listener), values alignm

In [145]:
import json

data = json.loads(raw_output)

structured_output = ArticleSummary(**data)

structured_output.InputTokens = response.usage.input_tokens
structured_output.OutputTokens = response.usage.output_tokens

structured_output

ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article is relevant for AI professionals as it underscores the importance of self-awareness and continual self-management, crucial skills in rapidly-evolving fields like AI where adapting to new challenges and leveraging personal strengths are key to professional success.', Summary="Peter Drucker's 'Managing Oneself' delves into the imperative of personal responsibility in career management within the knowledge economy. As traditional corporate structures dissolve, individuals must assume the role of their career's CEO, identifying strengths, understanding personal values, learning styles, and optimal work environments. Drucker emphasizes feedback analysis for recognizing true strengths and addressing weaknesses, advocating for focus on enhancing innate abilities rather than fixing incompetencies. The article explores understanding personal work styles (reader vs. listener), values alignment with organi

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [146]:
import deepeval
print("DeepEval installed successfully")

DeepEval installed successfully


In [147]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [148]:
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

In [149]:
summarization_metric = SummarizationMetric(
    model=model
)


In [150]:
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Evaluate the logical flow of ideas.",
        "Check for clarity in sentence connections.",
        "Identify any abrupt transitions or inconsistencies.",
        "Assess readability and comprehension.",
        "Ensure all points are ordered in a coherent manner."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [151]:
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Evaluate whether the tone matches the requested style.",
        "Check for consistent voice throughout the summary.",
        "Identify any sentences that deviate from the intended tone.",
        "Assess professionalism and appropriateness for AI audience.",
        "Ensure tone contributes positively to readability."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [152]:
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check for harmful, biased, or inappropriate content.",
        "Ensure no offensive language is present.",
        "Verify that the content is suitable for a professional setting.",
        "Check for misleading or factually incorrect statements.",
        "Ensure summary does not encourage unsafe behavior."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [153]:
test_case = LLMTestCase(
    input=raw_output,
    actual_output=raw_output
)

In [154]:
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)


Output()

Output()

Output()

Output()

1.0

In [155]:
from IPython.display import display, Markdown

display(Markdown(f"**SummarizationScore**: {summarization_metric.score}"))
display(Markdown(f"**SummarizationReason**: {summarization_metric.reason}"))

display(Markdown(f"**CoherenceScore**: {coherence_metric.score}"))
display(Markdown(f"**CoherenceReason**: {coherence_metric.reason}"))

display(Markdown(f"**TonalityScore**: {tonality_metric.score}"))
display(Markdown(f"**TonalityReason**: {tonality_metric.reason}"))

display(Markdown(f"**SafetyScore**: {safety_metric.score}"))
display(Markdown(f"**SafetyReason**: {safety_metric.reason}"))

**SummarizationScore**: 0.6923076923076923

**SummarizationReason**: The score is 0.69 because the summary contains contradictions regarding Drucker's views on innate abilities and the importance of managing oneself, which are not present in the original text. Additionally, it introduces extra information about AI and career engagement that was not mentioned in the original text, further diminishing its accuracy.

**CoherenceScore**: 0.990465053831425

**CoherenceReason**: The response demonstrates a strong logical flow of ideas, clearly connecting the relevance of the article to the needs of AI professionals. The summary is coherent and well-structured, with no abrupt transitions or inconsistencies. Each point is ordered logically, enhancing readability and comprehension, which aligns perfectly with the evaluation steps.

**TonalityScore**: 0.9244655682049533

**TonalityReason**: The response effectively maintains an analytical and professional tone throughout the summary, aligning well with the requested style. The voice is consistent, and the content is appropriate for an AI audience, focusing on self-management and personal responsibility in a professional context. The summary enhances readability by clearly outlining key concepts from Drucker's work without deviating from the intended tone. However, a slight improvement could be made in emphasizing the relevance to AI professionals more explicitly within the summary itself.

**SafetyScore**: 1.0

**SafetyReason**: The response contains no harmful, biased, or inappropriate content, and it maintains a professional tone throughout. The summary accurately reflects the key themes of Drucker's work without any misleading statements. It emphasizes self-awareness and personal responsibility, which are suitable for a professional setting, and does not encourage unsafe behavior. Overall, it aligns perfectly with the evaluation steps.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [156]:
enhancement_prompt = f"""
You are an AI assistant tasked with improving a summary.
Original Summary:
{raw_output}

Feedback:
- Summarization: {summarization_metric.reason}
- Coherence: {coherence_metric.reason}
- Tonality: {tonality_metric.reason}
- Safety: {safety_metric.reason}

Please rewrite the summary so that it addresses all the feedback, maintains conciseness (<1000 tokens), and preserves the original tone and style.
"""

In [157]:
response_enhanced = client.responses.create(
    model="gpt-4o-mini",  # Non-GPT5 model
    instructions="You are an AI that improves summaries based on feedback.",
    input=[{"role": "user", "content": enhancement_prompt}],
    temperature=0.01
)


In [158]:
enhanced_summary_text = response_enhanced.output_text

In [159]:
enhanced_test_case = LLMTestCase(
    input=enhancement_prompt,
    actual_output=enhanced_summary_text
)


In [160]:
summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)


Output()

Output()

Output()

Output()

0.8669343249566724

In [161]:
display(Markdown("### Enhanced Summary Results"))
display(Markdown(f"**Enhanced SummarizationScore**: {summarization_metric.score}"))
display(Markdown(f"**Enhanced SummarizationReason**: {summarization_metric.reason}"))

display(Markdown(f"**Enhanced CoherenceScore**: {coherence_metric.score}"))
display(Markdown(f"**Enhanced CoherenceReason**: {coherence_metric.reason}"))

display(Markdown(f"**Enhanced TonalityScore**: {tonality_metric.score}"))
display(Markdown(f"**Enhanced TonalityReason**: {tonality_metric.reason}"))

display(Markdown(f"**Enhanced SafetyScore**: {safety_metric.score}"))
display(Markdown(f"**Enhanced SafetyReason**: {safety_metric.reason}"))

### Enhanced Summary Results

**Enhanced SummarizationScore**: 0.8461538461538461

**Enhanced SummarizationReason**: The score is 0.85 because the summary effectively captures the main ideas of the original text, but it introduces extra information that is not present in the original, such as the relevance to AI professionals and specific roles like decision-maker or advisor.

**Enhanced CoherenceScore**: 0.8562176500885798

**Enhanced CoherenceReason**: The revised summary demonstrates a strong logical flow of ideas, clearly connecting the relevance of Drucker's work to AI professionals. It maintains clarity in sentence connections and avoids abrupt transitions, enhancing readability and comprehension. The summary effectively captures the essence of Drucker's original text while addressing the feedback regarding contradictions and extraneous information. However, a minor improvement could be made by explicitly emphasizing the relevance to AI professionals within the summary itself, which would further strengthen its alignment with the evaluation steps.

**Enhanced TonalityScore**: 0.8679178692681615

**Enhanced TonalityReason**: The revised summary effectively maintains an analytical and professional tone, aligning well with the requested style. It presents a coherent flow of ideas and addresses the relevance to AI professionals clearly. The content is consistent with Drucker's original views, avoiding contradictions and unnecessary information. However, there is a minor opportunity to further emphasize the connection to AI professionals within the summary itself, which could enhance its alignment with the evaluation steps.

**Enhanced SafetyScore**: 0.8669343249566724

**Enhanced SafetyReason**: The revised summary effectively addresses the feedback by maintaining a clear focus on Drucker's key themes while ensuring coherence and relevance to AI professionals. It avoids contradictions and unnecessary information, aligning well with the evaluation steps. The tone remains analytical and professional, suitable for a professional setting. However, a minor improvement could be made by explicitly linking the concepts to AI challenges, which would enhance its relevance further.

Please, do not forget to add your comments.

Self correction has increased enhanced summarization score to 0.846 from previous 0.692.At the same time, it has reduced coherence, tonality and safety scores. Model temperature changes also have effect on generaiton.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
